# ML Engine and GPU Kernel Functions From Scratch

###Phase 1: Implement a dynamic computational graph engine for backpropagation. The engine should support GPU, using CuPy and Triton.

Minimum Deliverable:

*   Tensor Class
*   Relu
*   Sigmoid
*   MSE
*   Cross Entropy
*   SGD
*   Adam
*   Batch Norm
*   Vector Addition
*   GEMM
*   Reduction Kernels

Stretch Deliverable:
*   Layer Norm
*   Dropout
*   Flash Attention


### Phase 2: Training a model

Using your engine, implement ResNet18, and train it on the ImageNet dataset.

Stretch goal: Implement ChatGPT2 using flash attention.

Minimum Deliverable:

ResNet 18

Stretch Deliverable:

ChatGPT 2

#Group A: Machine Learning Framework

Any arrays that you use should be done in cupy and numpy

Deliverables:
* Tensor Class
* Backpropagation Function
* Optimizer Class


###Tensor Class:

The tensor class should have the instance variables: name, value, parents, and derivative.

It shold have the methods: add, subtract, multiply, power, matrix_multiply, dot_product, sum, mean, relu, and sigmoid.

Hint: It may be helpful to construct an edge class that stores the derivative function that connects the nested variables.

###Backpropagation Function:

Backpropagation starts off with finding a topological ordering of the variables.

This topological ordering can be found with a post order dfs.

After you have a topological ordering, implement the forward method that computes the values of all of the variables.

Next, do the backward pass which goes backward through the ordering and computes derivatives.

###Optimizer:

You need a way to update the values of the input tensors. Implement a datastructure that points to the input variables of the computational graph. This class should have a step method that tells the optimizer how to modify the variables.

You should implement two optimizers

1) Stochastic Gradient Descent:

* https://ds100.org/sp26/lectures/13/

* https://ds100.org/sp26/lectures/14/


2) Adam

* https://eecs189.org/fa25/lecture/lec13/


# Group B: Kernel Functions

Background: It is easy to take a lot of the math that our computers do for granted. There are actual functions that people programmed that drive all of the math. However, most functions that you have worked with were implemented on CPU. Numpy arrays are computed by your cpu. In recent times, libraries like Pytorch can run on your GPU. The GPU computes things in parallel, turning sequential operations into things that can be computed simultaneously.

In some cases, you can turn things that would originally take O(n) time in to things that take O(1) time.

Your job is to write these math functions in **Triton**, the python library for GPU programming.

Triton Documentation: https://triton-lang.org/main/index.html

Deliverables:

* Vector Addition
* Mean
* GeMM
* Relu
* Fused Softmax
* Batch Norm
* Max Pooling
* Global Average Pooling


Possibly Helpful Youtube Tutorial: The dude has a playlist. I watched some of his videos but he's kind of annoying...

https://www.youtube.com/watch?v=FtmnriHLbAg&list=PL_NMDNzkCbLR69QafQUa6yTx-T-VPIoaZ


In [ ]:
import torch
import triton
import triton.language as tl


@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < n
    tl.store(out_ptr + offs,
             tl.load(x_ptr + offs, mask=mask) + tl.load(y_ptr + offs, mask=mask),
             mask=mask)


def vector_add(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    assert x.is_cuda and y.is_cuda, "inputs must be CUDA tensors"
    assert x.shape == y.shape
    out = torch.empty_like(x)
    n = x.numel()
    BLOCK_SIZE = 1024
    grid = (triton.cdiv(n, BLOCK_SIZE),)
    add_kernel[grid](x, y, out, n, BLOCK_SIZE=BLOCK_SIZE)
    return out


# --- usage ---
n = 2 ** 20  # 1M elements
x = torch.rand(n, device='cuda', dtype=torch.float32)
y = torch.rand(n, device='cuda', dtype=torch.float32)
out = vector_add(x, y)

# verify
assert torch.allclose(out, x + y)
print("max error:", (out - (x + y)).abs().max().item())
